In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## STEP 0 - Load a document ##

In [2]:
# STEP 0- loading a document to be chunked for embedding and vector storage
from pypdf import PdfReader

reader = PdfReader("../2_RAG/NovaS.pdf")
  
pages = [
    {"page_content": page.extract_text() or "", 
     "metadata": {"page": i, "source": "./2_RAG/NovaS.pdf"}
     }
    for i, page in enumerate(reader.pages)
]

#get the full content of the document
full_content = " ".join(doc["page_content"] for doc in pages)
full_content

'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting solutions to make better business decisions. \nDuring the first year of operations, the company worked mostly with local startups that did \nnot have large 

In [3]:
#STEP 1 - splitting the document into chunks using text splitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
chunker = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"), 
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=60
)
semantic_chunks = chunker.create_documents([full_content])
semantic_chunks

C:\Users\jethw\AppData\Local\Temp\ipykernel_25472\4122089546.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


[Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'),
 Document(metadata={}, page_content='At the \nbeginning, the company did not have large investments or a big office.'),
 Document(metadata={}, page_content='Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight.'),
 Document(metadata={}, page_content='Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting 

In [5]:
#STEP 2 & 3 - Create embeddings and store them in a vector database
# from langchain_chroma import Chroma
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
# Create a Chroma vector store from the chunk embeddings and the corresponding chunks
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=semantic_chunks,
    embedding=embed_model,
    persist_directory="./chroma_db_semantic"
)

In [7]:
## Step 4 - Connection to the vector database and retrieval of relevant chunks based on a query
vectorstore_conn = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


In [8]:
## Step 5 - Create a retriever from the vectorstore connection and use it to retrieve relevant chunks based on a query
# retriever = vectorstore_conn.as_retriever(search_type="similarity", search_kwargs={"k": 2})
#do similarity search on the vectorstore connection using the query "What is NovaS?" and retrieve the top 2 most similar chunks
query = "When was NovaSphere Technologies was founded?"
retrieved_chunks = vectorstore_conn.similarity_search(query, k=2)
retrieved_chunks

[Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'),
 Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.')]

In [33]:
## Step 6 - LLM Integration and Answer Generation
# Import the necessary libraries for LLM integration and answer generation. Use the retrieved chunks to generate a response to the query using a language model.
from langchain_openai import ChatOpenAI
llm_model = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

#form the query which yoou want to ask the LLM model. 
query = input() #"By 2019, how many employees were there?"

#retrieve the relevant chunks based on the query using the vectorstore connection. The retrieved chunks will be used as context for the LLM model to generate a response.
relevant_chunks = vectorstore_conn.similarity_search(query, k=3)

#fetch the context from the retrieved chunks to provide to the LLM model for generating a response. Fetch from the page_content of each chunk.
context = "\n".join([chunk.page_content for chunk in relevant_chunks])
print(f"Context: {context}")
# Pass the query and the context to the LLM model to generate a response. The LLM model will use the context to generate a response to the query.
llm_response = llm_model.invoke(f"{query}, answer the question based on the context provided. Context: {context}")
llm_response
# llm_model.invoke(f"config={{'context': '{context}', 'query': '{query}'}}")


Context: By the beginning of 2019, the organization had grown to more than fifteen employees. This
During 2023, the organization experienced steady growth. The number of employees
number of employees increased again, and the company also started hiring people who


AIMessage(content=' By 2019, there were more than fifteen employees in the organization.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 79, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ET6PgyBdryysTI1hJTCjw00t4j9Xu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0e85e-0697-7450-a728-a12afde43a22-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output_tokens': 15, 'total_tokens': 94, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})